In [18]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver

In [19]:
load_dotenv()

True

In [20]:
model = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash')

In [21]:
class JokeState(TypedDict):
    topic: str
    joke: str
    exp: str

In [22]:
def gen_joke(state: JokeState) -> JokeState:
    prompt = f"Generate a joke on the topic \n {state['topic']}"
    joke = model.invoke(prompt).content
    return {'joke': joke}

In [23]:
def gen_explaination(state: JokeState) -> JokeState:
    prompt = f"Explain the {state['joke']} on the topic\n {state['topic']}"
    exp = model.invoke(prompt).content
    return {'exp': exp}

In [24]:
graph = StateGraph(JokeState)

graph.add_node('gen_joke', gen_joke)
graph.add_node('gen_explaination', gen_explaination)

graph.add_edge(START, 'gen_joke')
graph.add_edge('gen_joke', 'gen_explaination')
graph.add_edge('gen_explaination', END)

chkpointer = MemorySaver()

workflow = graph.compile(checkpointer= chkpointer)

In [25]:
thread_id = '1'
config = {'configurable': {'thread_id': thread_id}}
response = workflow.invoke({'topic': 'AI'}, config= config)
response

{'topic': 'AI',
 'joke': 'Why did the AI break up with the calculator?\n\nBecause it said, "You\'re too binary for me! I need a partner who can process complex emotions, not just numbers."',
 'exp': 'This joke is a clever play on words and a humorous take on the capabilities of Artificial Intelligence versus a simple calculator. Here\'s why it\'s funny:\n\n1.  **Anthropomorphism:** The joke gives human-like qualities to both the AI and the calculator. AI is portrayed as having emotions and relationship needs, while the calculator is a "partner" that can be "broken up with." This unexpected humanization of machines is inherently funny.\n\n2.  **The Double Meaning of "Binary":** This is the core of the joke:\n    *   **Literal Meaning (for a calculator):** Calculators, and all computers at their fundamental level, operate using a **binary system** (0s and 1s). So, a calculator *is* literally "binary."\n    *   **Figurative Meaning (for a relationship/emotions):** In human terms, "binary"

# Time Travel

In [28]:
list(workflow.get_state_history(config = {'configurable': {'thread_id': thread_id}}))

[StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the AI break up with the calculator?\n\nBecause it said, "You\'re too binary for me! I need a partner who can process complex emotions, not just numbers."', 'exp': 'This joke is a clever play on words and a humorous take on the capabilities of Artificial Intelligence versus a simple calculator. Here\'s why it\'s funny:\n\n1.  **Anthropomorphism:** The joke gives human-like qualities to both the AI and the calculator. AI is portrayed as having emotions and relationship needs, while the calculator is a "partner" that can be "broken up with." This unexpected humanization of machines is inherently funny.\n\n2.  **The Double Meaning of "Binary":** This is the core of the joke:\n    *   **Literal Meaning (for a calculator):** Calculators, and all computers at their fundamental level, operate using a **binary system** (0s and 1s). So, a calculator *is* literally "binary."\n    *   **Figurative Meaning (for a relationship/emotions):** In h

In [29]:
workflow.invoke(None, config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f185fb3-783f-622b-8000-45b50a4a0ea7'}})

{'topic': 'AI',
 'joke': 'Why did the AI get kicked out of the comedy club?\n\nBecause its jokes were too *predictable*, and it kept explaining the optimal laughter-to-word ratio after every punchline!',
 'exp': 'This joke perfectly captures the comedic clash between the analytical, data-driven nature of AI and the spontaneous, human art of stand-up comedy. Here\'s why the AI got the boot:\n\n1.  **"Its jokes were too predictable."**\n    *   **AI Perspective:** AIs excel at pattern recognition. They can analyze millions of jokes, identify common structures, punchline formulas, and audience responses. While this allows them to *generate* jokes, it often means they produce variations of what\'s already known or statistically "safe."\n    *   **Comedy Reality:** Good comedy thrives on surprise, misdirection, and subverting expectations. If an audience can see the punchline coming from a mile away, the humor is lost. The AI, in its quest for optimal joke construction, likely ended up bein

# Update State

In [36]:
workflow.update_state(

        config={'configurable':{
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f185fb3-783f-622b-8000-45b50a4a0ea7'
            }
                },
        values={'topic': 'samosa'}
        )

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18602e-2960-64c7-8001-8be2d03dc23e'}}

In [37]:
workflow.invoke(None, {'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18602b-cf75-6b20-8001-363a8a4994f4'}})

{'topic': 'AI',
 'joke': 'Why did the AI break up with the calculator?\n\nBecause it said, "You\'re just not *adding up* to my expectations, and frankly, your data processing capabilities are *obsolete*."',
 'exp': 'This joke plays on the vast differences in capability and purpose between a modern Artificial Intelligence and a simple calculator, using a bit of anthropomorphism and wordplay.\n\nHere\'s why the AI broke up with the calculator:\n\n1.  **"You\'re just not *adding up* to my expectations."**\n    *   **The Pun:** "Adding up" has a double meaning here.\n        *   **Literal:** Calculators literally "add up" numbers. It\'s their primary function.\n        *   **Figurative:** In a relationship or professional context, "not adding up" means not meeting standards, not being good enough, or not being a suitable partner.\n    *   **AI\'s Perspective:** An AI\'s "expectations" are incredibly sophisticated. It deals with petabytes of data, complex algorithms, machine learning, natur

In [40]:
list(workflow.get_state_history(config= {'configurable': {'thread_id': '1'}}))

[StateSnapshot(values={'topic': 'AI', 'joke': 'Why did the AI break up with the calculator?\n\nBecause it said, "You\'re just not *adding up* to my expectations, and frankly, your data processing capabilities are *obsolete*."', 'exp': 'This joke plays on the vast differences in capability and purpose between a modern Artificial Intelligence and a simple calculator, using a bit of anthropomorphism and wordplay.\n\nHere\'s why the AI broke up with the calculator:\n\n1.  **"You\'re just not *adding up* to my expectations."**\n    *   **The Pun:** "Adding up" has a double meaning here.\n        *   **Literal:** Calculators literally "add up" numbers. It\'s their primary function.\n        *   **Figurative:** In a relationship or professional context, "not adding up" means not meeting standards, not being good enough, or not being a suitable partner.\n    *   **AI\'s Perspective:** An AI\'s "expectations" are incredibly sophisticated. It deals with petabytes of data, complex algorithms, mac

In [41]:
workflow.invoke(None, config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18602e-2960-64c7-8001-8be2d03dc23e'}})

{'topic': 'samosa',
 'joke': 'Why did the samosa get picked first for the secret mission?\n\nBecause it was always good at keeping its **fillings** to itself!',
 'exp': 'This is a classic food pun! Here\'s the breakdown:\n\n1.  **Samosa\'s "Fillings":** A samosa is a pastry with a savory filling (often potatoes, peas, spices, or meat) inside. The pastry shell effectively seals in its delicious contents, preventing them from spilling out.\n\n2.  **The Pun:** This is a play on words, as "fillings" sounds exactly like "feelings."\n\n3.  **"Keeping its feelings to itself":** If someone is good at "keeping their feelings to themselves," it means they are discreet, don\'t reveal their emotions easily, and are excellent at keeping secrets.\n\n**So, the joke works because:**\n\nFor a "secret mission," you need someone trustworthy who won\'t spill confidential information. The samosa is chosen because, literally, it keeps its **fillings** (the stuff inside it) contained, which sounds like it\'s